# South German Credit: data preprocessing

This notebook covers loading, column names, category meanings, feature types, missing values, duplicates, consistency checks, and findings.

## 1. Load and inspect the dataset
The path works when the notebook is run from the project root or the notebooks folder. Common missing-value markers are read as nulls.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 21)

raw_folder = Path('Dataset/raw')
if not raw_folder.exists():
    raw_folder = Path('../Dataset/raw')

missing_markers = ['?', 'NA', 'N/A', 'NaN', 'nan', 'NULL', 'null', 'None', 'none', 'missing', '-', '--']
original_df = pd.read_csv(
    raw_folder / 'SouthGermanCredit.asc',
    sep=r'\s+',
    na_values=missing_markers
)
df = original_df.copy()

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
df.info()

Shape: (1000, 21)
Columns: ['laufkont', 'laufzeit', 'moral', 'verw', 'hoehe', 'sparkont', 'beszeit', 'rate', 'famges', 'buerge', 'wohnzeit', 'verm', 'alter', 'weitkred', 'wohn', 'bishkred', 'beruf', 'pers', 'telef', 'gastarb', 'kredit']
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   laufkont  1000 non-null   int64
 1   laufzeit  1000 non-null   int64
 2   moral     1000 non-null   int64
 3   verw      1000 non-null   int64
 4   hoehe     1000 non-null   int64
 5   sparkont  1000 non-null   int64
 6   beszeit   1000 non-null   int64
 7   rate      1000 non-null   int64
 8   famges    1000 non-null   int64
 9   buerge    1000 non-null   int64
 10  wohnzeit  1000 non-null   int64
 11  verm      1000 non-null   int64
 12  alter     1000 non-null   int64
 13  weitkred  1000 non-null   int64
 14  wohn      1000 non-null   int64
 15  bishkred  1000 non-null   int64
 1

,laufkont,laufzeit,moral,verw,hoehe,sparkont,beszeit,rate,famges,buerge,wohnzeit,verm,alter,weitkred,wohn,bishkred,beruf,pers,telef,gastarb,kredit
0,1,18,4,2,1049,1,2,4,2,1,4,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,4,2,38,1,2,2,2,2,1,1,1


## 2. Rename columns
Use the 21 English names listed in the codetable. Pandas leaves any already descriptive column names unchanged.

In [2]:
rename_map = {'laufkont': 'status',
 'laufzeit': 'duration',
 'moral': 'credit_history',
 'verw': 'purpose',
 'hoehe': 'amount',
 'sparkont': 'savings',
 'beszeit': 'employment_duration',
 'rate': 'installment_rate',
 'famges': 'personal_status_sex',
 'buerge': 'other_debtors',
 'wohnzeit': 'present_residence',
 'verm': 'property',
 'alter': 'age',
 'weitkred': 'other_installment_plans',
 'wohn': 'housing',
 'bishkred': 'number_credits',
 'beruf': 'job',
 'pers': 'people_liable',
 'telef': 'telephone',
 'gastarb': 'foreign_worker',
 'kredit': 'credit_risk'}

df = df.rename(columns=rename_map)
display(df.head())

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,1,18,4,2,1049,1,2,4,2,1,4,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,4,2,38,1,2,2,2,2,1,1,1


## 3. Interpret encoded categorical values
The dictionaries below contain every category code and its meaning from `codetable.txt`. `df` keeps these codes. A readable EDA copy is created in section 7 after checking the data.

Important corrected codes:
- `credit_risk`: 0 = bad, 1 = good.
- `people_liable`: 1 = 3 or more, 2 = 0 to 2.
- `foreign_worker`: 1 = yes, 2 = no.

`personal_status_sex` combines categories, so sex and marital status cannot reliably be separated.

In [3]:
category_mappings = {'status': {1: 'no checking account',
            2: '... < 0 DM',
            3: '0<= ... < 200 DM',
            4: '... >= 200 DM / salary for at least 1 year'},
 'credit_history': {0: 'delay in paying off in the past',
                    1: 'critical account/other credits elsewhere',
                    2: 'no credits taken/all credits paid back duly',
                    3: 'existing credits paid back duly till now',
                    4: 'all credits at this bank paid back duly'},
 'purpose': {0: 'others',
             1: 'car (new)',
             2: 'car (used)',
             3: 'furniture/equipment',
             4: 'radio/television',
             5: 'domestic appliances',
             6: 'repairs',
             7: 'education',
             8: 'vacation',
             9: 'retraining',
             10: 'business'},
 'savings': {1: 'unknown/no savings account',
             2: '... <  100 DM',
             3: '100 <= ... <  500 DM',
             4: '500 <= ... < 1000 DM',
             5: '... >= 1000 DM'},
 'employment_duration': {1: 'unemployed',
                         2: '< 1 yr',
                         3: '1 <= ... < 4 yrs',
                         4: '4 <= ... < 7 yrs',
                         5: '>= 7 yrs'},
 'installment_rate': {1: '>= 35', 2: '25 <= ... < 35', 3: '20 <= ... < 25', 4: '< 20'},
 'personal_status_sex': {1: 'male : divorced/separated',
                         2: 'female : non-single or male : single',
                         3: 'male : married/widowed',
                         4: 'female : single'},
 'other_debtors': {1: 'none', 2: 'co-applicant', 3: 'guarantor'},
 'present_residence': {1: '< 1 yr', 2: '1 <= ... < 4 yrs', 3: '4 <= ... < 7 yrs', 4: '>= 7 yrs'},
 'property': {1: 'unknown / no property',
              2: 'car or other',
              3: 'building soc. savings agr./life insurance',
              4: 'real estate'},
 'other_installment_plans': {1: 'bank', 2: 'stores', 3: 'none'},
 'housing': {1: 'for free', 2: 'rent', 3: 'own'},
 'number_credits': {1: '1', 2: '2-3', 3: '4-5', 4: '>= 6'},
 'job': {1: 'unemployed/unskilled - non-resident',
         2: 'unskilled - resident',
         3: 'skilled employee/official',
         4: 'manager/self-empl./highly qualif. employee'},
 'people_liable': {1: '3 or more', 2: '0 to 2'},
 'telephone': {1: 'no', 2: 'yes (under customer name)'},
 'foreign_worker': {1: 'yes', 2: 'no'},
 'credit_risk': {0: 'bad', 1: 'good'}}

# Show the full code dictionary for reference.
for column, mapping in category_mappings.items():
    print(column)
    display(pd.Series(mapping, name='meaning').rename_axis('code').to_frame())

status
credit_history
purpose
savings
employment_duration
installment_rate
personal_status_sex
other_debtors
present_residence
property
other_installment_plans
housing
number_credits
job
people_liable
telephone
foreign_worker
credit_risk


,meaning
code,
1,no checking account
2,... < 0 DM
3,0<= ... < 200 DM
4,... >= 200 DM / salary for at least 1 year


,meaning
code,
0,delay in paying off in the past
1,critical account/other credits elsewhere
2,no credits taken/all credits paid back duly
3,existing credits paid back duly till now
4,all credits at this bank paid back duly


,meaning
code,
0,others
1,car (new)
2,car (used)
3,furniture/equipment
4,radio/television
5,domestic appliances
6,repairs
7,education
8,vacation


,meaning
code,
1,unknown/no savings account
2,... < 100 DM
3,100 <= ... < 500 DM
4,500 <= ... < 1000 DM
5,... >= 1000 DM


,meaning
code,
1,unemployed
2,< 1 yr
3,1 <= ... < 4 yrs
4,4 <= ... < 7 yrs
5,>= 7 yrs


... (9 outputs left out)

,meaning
code,
1,3 or more
2,0 to 2


,meaning
code,
1,no
2,yes (under customer name)


,meaning
code,
1,yes
2,no


,meaning
code,
0,bad
1,good


## 4. Identify feature types
There are 3 numerical, 11 nominal categorical, and 6 ordinal categorical predictors. `credit_risk` is the separate binary target.

- Numerical: actual measurements, rather than category codes.
- Nominal: categories without a required natural order. `status` and `savings` include special no-account or unknown categories, so they are kept nominal. `people_liable` is treated as binary nominal, consistent with the supplied R reader.
- Ordinal: employment tenure, installment percentage bands, residence duration, property hierarchy, grouped credit counts, and job quality. UCI classifies property and job as ordinal. Higher installment codes mean lower percentages; the code numbers are not literal percentages or equally spaced measurements.

In [4]:
numerical_features = ['duration', 'amount', 'age']

nominal_features = [
    'status', 'credit_history', 'purpose', 'savings', 'personal_status_sex',
    'other_debtors', 'other_installment_plans', 'housing', 'people_liable',
    'telephone', 'foreign_worker'
]

ordinal_features = [
    'employment_duration', 'installment_rate', 'present_residence',
    'property', 'number_credits', 'job'
]

target = 'credit_risk'

feature_types = pd.DataFrame({
    'feature': numerical_features + nominal_features + ordinal_features,
    'type': ['Numerical'] * 3 + ['Nominal categorical'] * 11 + ['Ordinal categorical'] * 6
})
display(feature_types)

,feature,type
0,duration,Numerical
1,amount,Numerical
2,age,Numerical
3,status,Nominal categorical
4,credit_history,Nominal categorical
5,purpose,Nominal categorical
6,savings,Nominal categorical
7,personal_status_sex,Nominal categorical
8,other_debtors,Nominal categorical
9,other_installment_plans,Nominal categorical


## 5. Check for missing values
Display null counts and unexpected text in columns that should contain numbers or numeric codes. Common missing markers were handled when loading the file. Undocumented numeric codes are checked in section 7.

Documented categories such as unknown/no savings account are valid information, not null values. Zero is also valid for some categories and the target.

As requested, fill nulls using each column's mean. No nulls occur in this file, so values remain unchanged. Means of category codes may not be valid categories; review the code-validation results if imputation is ever needed. An entirely missing column would have no mean.

In [5]:
missing_before = df.isna().sum()
display(missing_before.to_frame('missing_values'))

# Report unexpected text without silently replacing it.
for column in df.columns:
    numeric_values = pd.to_numeric(df[column], errors='coerce')
    unexpected_text = df.loc[df[column].notna() & numeric_values.isna(), column]
    if not unexpected_text.empty:
        print(column, 'unexpected values:', unexpected_text.unique())

# All columns in this dataset are stored as numbers.
df = df.fillna(df.mean(numeric_only=True))
print('Nulls filled:', missing_before.sum() - df.isna().sum().sum())
print('Remaining nulls:', df.isna().sum().sum())

Nulls filled: 0
Remaining nulls: 0


,missing_values
status,0
duration,0
credit_history,0
purpose,0
amount,0
savings,0
employment_duration,0
installment_rate,0
personal_status_sex,0
other_debtors,0


## 6. Check for duplicate records
Check complete rows in the original data before cleaning. A matching profile is not enough to prove a data-entry error because there is no unique applicant ID. Keep any matching records unless there is evidence that they are errors.

In [6]:
duplicate_count = original_df.duplicated().sum()
print('Extra fully duplicated rows:', duplicate_count)
display(original_df[original_df.duplicated(keep=False)])

# No duplicate rows are removed automatically.

Extra fully duplicated rows: 0


,laufkont,laufzeit,moral,verw,hoehe,sparkont,beszeit,rate,famges,buerge,wohnzeit,verm,alter,weitkred,wohn,bishkred,beruf,pers,telef,gastarb,kredit


## 7. Validate data consistency
Compare categorical values with the codetable. Report invalid codes and documented codes absent from this sample. Check numerical ranges and non-positive values, then confirm the target counts using 0 = bad and 1 = good.

Duration is in months and age is in years. The credit amount has undergone an unspecified monotonic transformation, so exact original monetary values cannot be recovered. Retain the supplied amounts and do not remove extreme values solely because they are large.

In [7]:
category_checks = []
for column, mapping in category_mappings.items():
    observed_codes = set(df[column].dropna())
    allowed_codes = set(mapping)
    category_checks.append({
        'feature': column,
        'invalid_codes': sorted(observed_codes - allowed_codes),
        'unused_codes': sorted(allowed_codes - observed_codes)
    })

category_validation = pd.DataFrame(category_checks)
display(category_validation)
display(df[numerical_features].describe())
print('Non-positive numerical values:')
display((df[numerical_features] <= 0).sum().to_frame('count'))

print('Target counts: 0 = bad, 1 = good')
display(df['credit_risk'].value_counts().sort_index().to_frame('count'))

Non-positive numerical values:
Target counts: 0 = bad, 1 = good


,feature,invalid_codes,unused_codes
0,status,[],[]
1,credit_history,[],[]
2,purpose,[],[7]
3,savings,[],[]
4,employment_duration,[],[]
5,installment_rate,[],[]
6,personal_status_sex,[],[]
7,other_debtors,[],[]
8,present_residence,[],[]
9,property,[],[]


,duration,amount,age
count,1000.000000,1000.00000,1000.00000
mean,20.903000,3271.24800,35.54200
std,12.058814,2822.75176,11.35267
min,4.000000,250.00000,19.00000
25%,12.000000,1365.50000,27.00000
50%,18.000000,2319.50000,33.00000
75%,24.000000,3972.25000,42.00000
max,72.000000,18424.00000,75.00000


,count
duration,0
amount,0
age,0


,count
credit_risk,
0,300
1,700


In [ ]:

eda_df = df.copy()
for column, mapping in category_mappings.items():
    labels = df[column].map(mapping)
    eda_df[column] = pd.Categorical(
        labels,
        categories=list(mapping.values()),
        ordered=column in ordinal_features
    )

display(eda_df.head())

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,no checking account,18,all credits at this bank paid back duly,car (used),1049,unknown/no savings account,< 1 yr,< 20,female : non-single or male : single,none,>= 7 yrs,car or other,21,none,for free,1,skilled employee/official,0 to 2,no,no,good
1,no checking account,9,all credits at this bank paid back duly,others,2799,unknown/no savings account,1 <= ... < 4 yrs,25 <= ... < 35,male : married/widowed,none,1 <= ... < 4 yrs,unknown / no property,36,none,for free,2-3,skilled employee/official,3 or more,no,no,good
2,... < 0 DM,12,no credits taken/all credits paid back duly,retraining,841,... < 100 DM,4 <= ... < 7 yrs,25 <= ... < 35,female : non-single or male : single,none,>= 7 yrs,unknown / no property,23,none,for free,1,unskilled - resident,0 to 2,no,no,good
3,no checking account,12,all credits at this bank paid back duly,others,2122,unknown/no savings account,1 <= ... < 4 yrs,20 <= ... < 25,male : married/widowed,none,1 <= ... < 4 yrs,unknown / no property,39,none,for free,2-3,unskilled - resident,3 or more,no,yes,good
4,no checking account,12,all credits at this bank paid back duly,others,2171,unknown/no savings account,1 <= ... < 4 yrs,< 20,male : married/widowed,none,>= 7 yrs,car or other,38,bank,rent,2-3,unskilled - resident,0 to 2,no,yes,good


## 8. Preprocessing findings

For the supplied dataset:
- The dataset contains 1,000 rows, 20 predictors, and 1 target.
- All 21 German column names were replaced with the official English names.
- Category mappings were taken from `codetable.txt`; readable labels are stored separately in `eda_df`.
- Predictors are classified as 3 numerical, 11 nominal categorical, and 6 ordinal categorical variables.
- No nulls, unexpected text values, or undocumented category codes were found. Mean imputation therefore changed no values.
- No fully duplicated rows were found, so no rows were removed.
- Numerical ranges are duration 4-72 months, amount 250-18,424 in the supplied transformed scale, and age 19-75 years. No non-positive values were found.
- The target contains 300 bad credits (0) and 700 good credits (1).
- Purpose code 7 (education) is documented but absent from this sample. It remains in the category dictionary.
- No value corrections, outlier removal, scaling, or model encoding were needed for this preprocessing audit.

The raw file is unchanged, and `original_df` retains the original loaded values and column names.

In [9]:
print('Original dataset shape:', original_df.shape)
print('Preprocessed dataset shape:', df.shape)
print('Readable EDA dataset shape:', eda_df.shape)

Original dataset shape: (1000, 21)
Preprocessed dataset shape: (1000, 21)
Readable EDA dataset shape: (1000, 21)
